In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # force CPU only

import tensorflow as tf
import keras
print(f"TF: {tf.__version__}, Keras: {keras.__version__}")
print(f"GPUs visible: {tf.config.list_physical_devices('GPU')}")  # should print []import tensorflow as tf
import keras
import numpy as np
import pandas as pd
import pickle
import os
from PIL import Image
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

print(f"TF: {tf.__version__}, Keras: {keras.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

with open('../data/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

CLASSES    = config['classes']
IMAGE_SIZE = config['image_size']
test_df    = config['test_df']
SEG_DIR    = '../data/segmented_images'

print(f"Classes: {len(CLASSES)}")
print(f"Test set: {len(test_df):,}")

I0000 00:00:1784276786.006585   19917 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF: 2.21.0, Keras: 3.15.0
GPUs visible: []
TF: 2.21.0, Keras: 3.15.0
GPUs: []
Classes: 15
Test set: 22,424


E0000 00:00:1784276794.406854   19917 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
I0000 00:00:1784276794.406943   19917 cuda_diagnostics.cc:160] env: CUDA_VISIBLE_DEVICES="-1"
I0000 00:00:1784276794.406961   19917 cuda_diagnostics.cc:163] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
I0000 00:00:1784276794.406975   19917 cuda_diagnostics.cc:171] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
I0000 00:00:1784276794.406981   19917 cuda_diagnostics.cc:176] retrieving CUDA diagnostic information for host: teesta
I0000 00:00:1784276794.406989   19917 cuda_diagnostics.cc:183] hostname: teesta
I0000 00:00:1784276794.407136   19917 cuda_diagnostics.cc:190] libcuda reported version is: 530.30.2
I0000 00:00:1784276794.407167   19917 cuda_diagnostics.cc:194] kernel reported 

In [2]:
import json

MODEL_DIR = '../models/hybrid_convnext_vgg'

with open(os.path.join(MODEL_DIR, 'config.json')) as f:
    model_config_dict = json.load(f)

# compile_config lives at the TOP level, not nested inside 'config'
if 'compile_config' in model_config_dict:
    del model_config_dict['compile_config']

model = keras.models.model_from_json(json.dumps(model_config_dict))
model.load_weights(os.path.join(MODEL_DIR, 'model.weights.h5'))

model.summary()
print(f"\n✅ Loaded — input: {model.input_shape}, output: {model.output_shape}")

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_pres… │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stem  │ (None, 56, 56,    │      6,528 │ convnext_base_pr… │
│ (Sequential)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │      6,400 │ convnext_base_st… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │        256 │ convnext_base_st… │
│ (LayerNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │     66,048 │ convnext_base_st… │
│ (Dense)             │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │          0 │ convnext_base_st… │
│ (Activation)        │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │     65,664 │ convnext_base_st… │
│ (Dense)             │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │        128 │ convnext_base_st… │
│ (LayerScale)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │          0 │ convnext_base_st… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 56, 56,    │          0 │ convnext_base_st… │
│                     │ 128)              │            │ convnext_base_st… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │      6,400 │ add[0][0]         │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │        256 │ convnext_base_st… │
│ (LayerNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │     66,048 │ convnext_base_st… │
│ (Dense)             │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │          0 │ convnext_base_st… │
│ (Activation)        │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │     65,664 │ convnext_base_st… │
│ (Dense)             │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ convnext_base_stag… │ (None, 56, 56,    │        128 │ convnext_base_st

 Total params: 102,682,575 (391.70 MB)

 Trainable params: 11,679,247 (44.55 MB)

 Non-trainable params: 91,003,328 (347.15 MB)


✅ Loaded — input: (None, 224, 224, 3), output: (None, 15)


In [3]:
for c in CLASSES:
    if c not in test_df.columns:
        test_df[c] = test_df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

def load_image(fname):
    img_path = tf.strings.join([SEG_DIR + '/', fname])
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    return img

fnames = test_df['Image Index'].tolist()
labels = test_df[CLASSES].values.astype(np.float32)

test_ds = tf.data.Dataset.from_tensor_slices((fnames, labels))
test_ds = test_ds.map(lambda f, l: (load_image(f), l), num_parallel_calls=tf.data.AUTOTUNE)
test_ds_batched = test_ds.batch(64).prefetch(tf.data.AUTOTUNE)

print(f"tf.data pipeline ready - {len(fnames)} test images")
print(f"Labels shape: {labels.shape}")

tf.data pipeline ready - 22424 test images
Labels shape: (22424, 15)


In [4]:
print("Running predictions on test set...")
y_pred_probs = model.predict(test_ds_batched, verbose=1)
y_true = labels

print("Predictions done!")
print(f"Predictions shape: {y_pred_probs.shape}")
print(f"True labels shape: {y_true.shape}")

Running predictions on test set...


I0000 00:00:1784276877.407785   20841 service.cc:153] XLA service 0x7fbc04015c70 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784276877.407875   20841 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1784276877.657629   20841 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


351/351 ━━━━━━━━━━━━━━━━━━━━ 5875s 17s/step
Predictions done!
Predictions shape: (22424, 15)
True labels shape: (22424, 15)


In [7]:
np.save('../data/hybrid_convnext_vgg_predictions.npy', y_pred_probs)
np.save('../data/hybrid_convnext_vgg_true_labels.npy', y_true)
print("Predictions saved to disk!")

Predictions saved to disk!


In [8]:
import wandb

wandb.init(
    project='xray-classification',
    name='04f-hybrid-convnext-vgg16-eval',
    config={
        'model': 'Hybrid_ConvNeXt_VGG16',
        'source': 'external_pretrained',
        'framework': 'keras',
        'total_params': 102682575,
        'trainable_params': 11679247,
        'test_images': 22424,
        'note': 'evaluated on CPU due to GPU/CUDA driver mismatch'
    }
)
print("Wandb initialized!")

ImportError: cannot import name 'Imports' from 'wandb.proto.wandb_telemetry_pb2' (/home/lhotse1/student/btech2023/achanta/miniconda3/envs/keras_eval/lib/python3.11/site-packages/wandb/proto/wandb_telemetry_pb2.py)

In [1]:
import tensorflow as tf
import keras
import numpy as np
import pandas as pd
import pickle
import os
import wandb
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

with open('../data/data_config.pkl', 'rb') as f:
    config = pickle.load(f)
CLASSES = config['classes']
print("Setup done, wandb imported cleanly:", wandb.__version__)

I0000 00:00:1784283597.736730     935 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Setup done, wandb imported cleanly: 0.22.3


In [2]:
y_pred_probs = np.load('../data/hybrid_convnext_vgg_predictions.npy')
y_true = np.load('../data/hybrid_convnext_vgg_true_labels.npy')
print(f"Reloaded — predictions: {y_pred_probs.shape}, labels: {y_true.shape}")

Reloaded — predictions: (22424, 15), labels: (22424, 15)


In [3]:
wandb.init(
    project='xray-classification',
    name='04f-hybrid-convnext-vgg16-eval',
    config={'model': 'Hybrid_ConvNeXt_VGG16', 'test_images': 22424}
)
print("Wandb initialized!")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: chandinigunna06 (chandinigunna06-iiest-shibpur) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb initialized!


In [4]:
auc_per_class = {}
for i, c in enumerate(CLASSES):
    try:
        auc_per_class[c] = roc_auc_score(y_true[:, i], y_pred_probs[:, i])
    except ValueError:
        auc_per_class[c] = float('nan')

mean_auc = np.nanmean(list(auc_per_class.values()))
y_pred_binary = (y_pred_probs > 0.5).astype(int)
f1_macro = f1_score(y_true, y_pred_binary, average='macro', zero_division=0)
overall_accuracy = accuracy_score(y_true.flatten(), y_pred_binary.flatten())

print(f"Mean AUC: {mean_auc:.4f}")
print(f"F1 macro: {f1_macro:.4f}")
print(f"Per-label accuracy: {overall_accuracy:.4f}")

Mean AUC: 0.5371
F1 macro: 0.0469
Per-label accuracy: 0.9221


In [6]:
with open('../data/data_config.pkl', 'rb') as f:
    config = pickle.load(f)

CLASSES    = config['classes']
IMAGE_SIZE = config['image_size']
test_df    = config['test_df']
SEG_DIR    = '../data/segmented_images'

for c in CLASSES:
    if c not in test_df.columns:
        test_df[c] = test_df['Finding Labels'].apply(lambda x: 1 if c in str(x).split('|') else 0)

fnames = test_df['Image Index'].tolist()
labels = test_df[CLASSES].values.astype(np.float32)

print(f"Rebuilt — {len(fnames)} filenames, labels shape: {labels.shape}")

Rebuilt — 22424 filenames, labels shape: (22424, 15)


In [7]:
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

sample_idx = np.random.RandomState(42).choice(len(fnames), 500, replace=False)
sample_fnames = [fnames[i] for i in sample_idx]
sample_labels = labels[sample_idx]

def load_image_vgg_preprocessed(fname):
    img_path = tf.strings.join([SEG_DIR + '/', fname])
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = vgg_preprocess(img)
    return img

sample_ds = tf.data.Dataset.from_tensor_slices((sample_fnames, sample_labels))
sample_ds = sample_ds.map(lambda f, l: (load_image_vgg_preprocessed(f), l), num_parallel_calls=tf.data.AUTOTUNE)
sample_ds_batched = sample_ds.batch(32).prefetch(tf.data.AUTOTUNE)

print("Testing with VGG-style preprocessing on 500 samples...")
y_pred_sample = model.predict(sample_ds_batched, verbose=1)

sample_auc = []
for i in range(len(CLASSES)):
    try:
        sample_auc.append(roc_auc_score(sample_labels[:, i], y_pred_sample[:, i]))
    except ValueError:
        pass

print(f"Sample mean AUC with VGG preprocessing: {np.nanmean(sample_auc):.4f}")

Testing with VGG-style preprocessing on 500 samples...


I0000 00:00:1784283944.477799     935 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 30953 MB memory:  -> device: 0, name: Tesla V100-PCIE-32GB, pci bus id: 0000:3b:00.0, compute capability: 7.0
I0000 00:00:1784283944.478929     935 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 30953 MB memory:  -> device: 1, name: Tesla V100-PCIE-32GB, pci bus id: 0000:af:00.0, compute capability: 7.0


NameError: name 'model' is not defined

In [8]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # force CPU to avoid the earlier GPU crash

import json

MODEL_DIR = '../models/hybrid_convnext_vgg'

with open(os.path.join(MODEL_DIR, 'config.json')) as f:
    model_config_dict = json.load(f)

if 'compile_config' in model_config_dict:
    del model_config_dict['compile_config']

model = keras.models.model_from_json(json.dumps(model_config_dict))
model.load_weights(os.path.join(MODEL_DIR, 'model.weights.h5'))

print(f"Model reloaded — input: {model.input_shape}, output: {model.output_shape}")

Model reloaded — input: (None, 224, 224, 3), output: (None, 15)


In [ ]:
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess

sample_idx = np.random.RandomState(42).choice(len(fnames), 500, replace=False)
sample_fnames = [fnames[i] for i in sample_idx]
sample_labels = labels[sample_idx]

def load_image_vgg_preprocessed(fname):
    img_path = tf.strings.join([SEG_DIR + '/', fname])
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
    img = vgg_preprocess(img)
    return img

sample_ds = tf.data.Dataset.from_tensor_slices((sample_fnames, sample_labels))
sample_ds = sample_ds.map(lambda f, l: (load_image_vgg_preprocessed(f), l), num_parallel_calls=tf.data.AUTOTUNE)
sample_ds_batched = sample_ds.batch(32).prefetch(tf.data.AUTOTUNE)

print("Testing with VGG-style preprocessing on 500 samples...")
y_pred_sample = model.predict(sample_ds_batched, verbose=1)

sample_auc = []
for i in range(len(CLASSES)):
    try:
        sample_auc.append(roc_auc_score(sample_labels[:, i], y_pred_sample[:, i]))
    except ValueError:
        pass

print(f"Sample mean AUC with VGG preprocessing: {np.nanmean(sample_auc):.4f}")

Testing with VGG-style preprocessing on 500 samples...


I0000 00:00:1784285055.408923    2609 service.cc:153] XLA service 0x7fd8040719e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1784285055.408980    2609 service.cc:161]   StreamExecutor [0]: Tesla V100-PCIE-32GB, Compute Capability 7.0 (Driver: 12.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1784285055.408994    2609 service.cc:161]   StreamExecutor [1]: Tesla V100-PCIE-32GB, Compute Capability 7.0 (Driver: 12.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1784285055.813958    2609 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1784285057.226162    2609 cuda_dnn.cc:461] Loaded cuDNN version 92400
